# 02c · Our own multilingual embedding model (English, Hindi, French), pretrained on this data

02b fine-tunes `multilingual-e5-small`. This notebook instead builds an embedding model **from scratch**, trained
only on the competition's records, with its own tokenizer, in two stages on 2 × T4.

1. **Text.** Exactly 02 full's text: `query: name | address`, with Indian-script names and addresses
   transliterated and names passed through the word map. The model can then replace e5 in 02 full as is. For
   records written in an Indian script (Hindi and the other scripts in the data), the original text is used too.
2. **Tokenizer.** WordPiece, 32,000 tokens, learned from the records. It covers English, the Indian scripts in the
   data and French, and gives whole tokens to this data's frequent words (`pvt`, `ltd`, `nagar`, `sarl`, `rue`...).
3. **Stage 1: masked-language-model pretraining (MLM).** A small BERT (6 layers, 384 wide, about 23M parameters)
   learns to fill in hidden tokens, on up to 4M texts per country. Contrastive training from random weights
   learns poorly; MLM first teaches it the words, scripts and spelling variants.
4. **Stage 2: contrastive training.** `MultipleNegativesRankingLoss`: each text must be closer to its positive
   than to every other text of the batch. The batch is shared by both GPUs, so each text competes with all 512
   rows of the step. The pairs:

   | Pairs | What | Countries |
   |---|---|---|
   | `matches` | S1 ↔ one of its true S2/S3 records, plus a lookalike from its bucket as hard negative (as 02b). Only S1 **outside** LightGBM's training set | India, US |
   | `noisy` | a record ↔ a copy with noise like the data's: typos, abbreviations (road → rd, rue → r), dropped or moved address words, legal form removed, duplicated words, name written as a website | India, US, France |
   | `script` | a record's transliterated text ↔ its original Indian-script text | India |
   | `script_matches` | S1 ↔ its true record in its original Indian script | India |

   France has no labels. The noisy copies of its records are its only contrastive signal.
5. **Evaluation.** e5-style top-k recall at full size, with 02b's code, on S1 that LightGBM trains on. This model
   never trained on their pairs. It is compared with `multilingual-e5-small` and with any model in
   `COMPARE_MODELS`, such as 02b's fine-tuned e5.

**Test text.** The tokenizer, the MLM stage and the noisy copies also use the test records' **text**. France exists
only in the test set, so without it the model would never see French. No test label exists, and none is used. This
is the same kind of use as 03's word rarity, which is computed over the test S1. Check it against the competition
rules before submitting anything built on this model. `USE_TEST_TEXT=false` trains on train text only (and then
the model sees no French).

**No leakage into LightGBM.** The `matches` and `script_matches` pairs never include the 200,000 S1 per country that
03 full trains LightGBM on (seed 42, recomputed as in 02b). MLM and noisy copies use no labels.

---

**Inputs (attach on Kaggle)**

| Input | Needed for |
|---|---|
| 01's `processed/` (`FULL_RUN=true`) | required: the records' text (train and test) |
| competition data (`train_ground_truth.tsv`) | required: the true pairs |
| 02 full's output (`buckets/train_*.parquet`, `translit_wordmap.json`) | recommended: hard negatives and the exact word map |

**Outputs** (`/kaggle/working/er_embed/`)

| File | Content |
|---|---|
| `er-embed-small/` | the final model (SentenceTransformer format, drop-in for 02 full) |
| `er-embed-small-mlm/` | the stage-1 model and tokenizer (to rerun stage 2 without stage 1: `MLM_MODEL`) |
| `eval_recall.csv` | recall@5/10/30: our model vs e5 (and `COMPARE_MODELS`) |
| `mlm_log.csv`, `contrastive_log.csv`, `embed_config.json` | loss curves and settings |
| `lgbm_train_s1.txt`, `matches_s1.txt`, `translit_wordmap.json` | LightGBM's S1 (never trained on), the S1 of the `matches` pairs, the word map |

**Run on Kaggle:** accelerator **GPU T4 x2**, internet on (the e5 comparison downloads it).
1. **Quick check first:** `QUICK_RUN=true`, either as an environment variable or by editing the line in the next
   cell. It runs every step on small samples in about 15-20 minutes, so a mistake shows up before the long run.
2. **Full run:** `QUICK_RUN=false`. My estimate is 3-4 hours: data preparation ~30 min, stage 1 ~45 min, stage 2
   ~60 min, evaluation ~30 min. The training steps print the time left.

**Using the model:** add this notebook's output as a dataset, then:
1. Run 02 full with `E5_MODELS=/kaggle/input/<dataset>/er_embed/er-embed-small`. Its buckets go to
   `embeddings_full/er-embed-small/`.
2. Run 03 full with `BUCKET_DIR=<that run>/embeddings_full/er-embed-small/buckets`, keeping its defaults
   `TRAIN_S1_PER_COUNTRY=200000` and seed 42.
3. Compare 03's cross-validation (0.9796 with e5-small) and 02 full's train report.

**Rules:** the model is our own (~23M parameters, trained from random weights on the competition data only). The
libraries are PyTorch (BSD), transformers, tokenizers and sentence-transformers (Apache 2.0).

## Settings
Each can also be set as an environment variable of the same name. `QUICK_RUN=true` replaces the sizes by the values
in brackets, unless the size itself is set.

| Setting | Default | Meaning |
|---|---|---|
| `QUICK_RUN` | `false` | small samples everywhere, to check the notebook end to end |
| `USE_TEST_TEXT` | `true` | also use the test records' text (no labels): tokenizer, MLM, noisy copies. Needed for French |
| `MODEL_NAME` | `er-embed-small` | output folder name (02 full names its buckets folder after it) |
| `VOCAB_SIZE`, `HIDDEN`, `LAYERS`, `HEADS`, `MAX_LEN` | `32000`, `384`, `6`, `6`, `128` | tokenizer and model size (`MAX_LEN` as 02 full) |
| `TOKENIZER_TEXTS` | `3000000` (200,000) | texts the tokenizer is learned from |
| `MLM_TEXTS_PER_COUNTRY`, `MLM_EPOCHS`, `MLM_BATCH`, `MLM_LR` | `4000000` (100,000), `1`, `1024` (256), `5e-4` | stage 1 |
| `PAIRS_PER_COUNTRY`, `HARD_NEG_TOP` | `1000000` (20,000), `10` | `matches` pairs per country, hard negative among the S1's 10 best non-matching candidates |
| `NOISY_PAIRS_PER_COUNTRY`, `SCRIPT_PAIRS` | `500000` (20,000), `300000` (5,000) | `noisy` pairs per country; `script` and `script_matches` pairs each |
| `CL_EPOCHS`, `CL_BATCH`, `CL_LR`, `SCALE` | `1`, `512` (128), `1e-4`, `20` | stage 2 (`CL_BATCH` = rows per step over all GPUs) |
| `LGBM_SEED`, `LGBM_TRAIN_S1_PER_COUNTRY` | `42`, `200000` | must equal 03 full's `SEED` and `TRAIN_S1_PER_COUNTRY` |
| `EVAL_COUNTRIES`, `EVAL_S1`, `SEEN_EVAL_S1` | `India`, `20000` (500), `5000` (200) | evaluation at full size |
| `COMPARE_MODELS` | `intfloat/multilingual-e5-small` | models evaluated next to ours (comma-separated names or folders; empty = none) |
| `TRAIN_GPUS` | all GPUs | training processes, one per GPU (DDP) |
| `MLM_MODEL` | empty | folder of an earlier stage-1 model (`er-embed-small-mlm`): skip the tokenizer and stage 1 |
| `TRAINED_MODEL` | empty | folder of an earlier final model: skip training, only evaluate |
| `EMBED_DIR`, `WORK_DIR` | `/kaggle/working/er_embed`, `/tmp/er_embed_work` | output; large temporary files (not kept) |

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "experiments":     # run from the repo root, like the pipeline notebooks (no-op on Kaggle)
    os.chdir(Path.cwd().parent)

import collections
import csv
import gc
import glob
import importlib.util
import inspect
import itertools
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("WANDB_DISABLED", "true")          # Kaggle ships wandb; the Trainer would otherwise ask for a key

for pkg in ("anyascii", "rapidfuzz", "datasets", "accelerate"):          # ISC, MIT, Apache 2.0, Apache 2.0
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import sentence_transformers  # noqa: E402
if int(sentence_transformers.__version__.split(".")[0]) < 3:     # the Trainer API of stage 2 needs >= 3.0
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0"])
    raise SystemExit("sentence-transformers was upgraded: restart the kernel and run all again")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyarrow as pa  # noqa: E402
import pyarrow.compute as pc  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402
import torch  # noqa: E402
from anyascii import anyascii  # noqa: E402  (ISC)
from rapidfuzz import fuzz  # noqa: E402  (MIT)
from rapidfuzz import process as rf_process  # noqa: E402
from sentence_transformers import SentenceTransformer  # noqa: E402  (Apache 2.0)
from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, processors, trainers  # noqa: E402
from transformers import PreTrainedTokenizerFast  # noqa: E402  (Apache 2.0)

QUICK_RUN = os.environ.get("QUICK_RUN", "false") == "true"


def setting(name, default, quick=None):
    """Environment variable `name` (cast like default), else the QUICK_RUN value, else the default."""
    if os.environ.get(name, "") != "":
        return type(default)(os.environ[name])
    return type(default)(quick) if QUICK_RUN and quick is not None else default


USE_TEST_TEXT = os.environ.get("USE_TEST_TEXT", "true") == "true"
MODEL_NAME = os.environ.get("MODEL_NAME", "er-embed-small")
VOCAB_SIZE = setting("VOCAB_SIZE", 32000)
HIDDEN, LAYERS, HEADS = setting("HIDDEN", 384), setting("LAYERS", 6), setting("HEADS", 6)
MAX_LEN = setting("MAX_LEN", 128)                # as 02 full's MAX_SEQ_LENGTH
TOKENIZER_TEXTS = setting("TOKENIZER_TEXTS", 3_000_000, 200_000)
MLM_TEXTS_PER_COUNTRY = setting("MLM_TEXTS_PER_COUNTRY", 4_000_000, 100_000)
MLM_EPOCHS = setting("MLM_EPOCHS", 1.0)
MLM_BATCH = setting("MLM_BATCH", 1024, 256)
MLM_LR = setting("MLM_LR", 5e-4)
MLM_PROB = 0.15
PAIRS_PER_COUNTRY = setting("PAIRS_PER_COUNTRY", 1_000_000, 20_000)
HARD_NEG_TOP = setting("HARD_NEG_TOP", 10)
NOISY_PAIRS_PER_COUNTRY = setting("NOISY_PAIRS_PER_COUNTRY", 500_000, 20_000)
SCRIPT_PAIRS = setting("SCRIPT_PAIRS", 300_000, 5_000)
CL_EPOCHS = setting("CL_EPOCHS", 1.0)
CL_BATCH = setting("CL_BATCH", 512, 128)
CL_LR = setting("CL_LR", 1e-4)
SCALE = setting("SCALE", 20.0)
LGBM_SEED = setting("LGBM_SEED", 42)                                        # 03 full's SEED
LGBM_TRAIN_S1_PER_COUNTRY = setting("LGBM_TRAIN_S1_PER_COUNTRY", 200_000)   # 03 full's TRAIN_S1_PER_COUNTRY
EVAL_COUNTRIES = [c.strip() for c in os.environ.get("EVAL_COUNTRIES", "India").split(",") if c.strip()]
EVAL_S1 = setting("EVAL_S1", 20_000, 500)
SEEN_EVAL_S1 = setting("SEEN_EVAL_S1", 5_000, 200)
EVAL_K = (5, 10, 30)
COMPARE_MODELS = [m.strip() for m in os.environ.get("COMPARE_MODELS", "intfloat/multilingual-e5-small").split(",")
                  if m.strip()]
MLM_MODEL = os.environ.get("MLM_MODEL", "")
TRAINED_MODEL = os.environ.get("TRAINED_MODEL", "")
QUICK_ROWS = 60_000 if QUICK_RUN else None       # quick run: rows read per file (plus the true matches of those S1)
SEED = 42
PREFIX = "query: "                               # 02 full puts it before every text, for every model
OUT_DIR = Path(os.environ.get("EMBED_DIR") or ("/kaggle/working/er_embed" if os.path.isdir("/kaggle/working")
                                               else "er_embed"))
WORK_DIR = Path(os.environ.get("WORK_DIR") or "/tmp/er_embed_work")
MODEL_DIR = OUT_DIR / MODEL_NAME
MLM_DIR = Path(MLM_MODEL) if MLM_MODEL else OUT_DIR / f"{MODEL_NAME}-mlm"
TOKENIZER_DIR = WORK_DIR / "tokenizer"
PAIR_DIR, CORPUS_DIR = WORK_DIR / "pairs", WORK_DIR / "corpus"
for d in (OUT_DIR, WORK_DIR, PAIR_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []
GPU_NAME = torch.cuda.get_device_name(0) if GPUS else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (as in 02 full)
USE_FP16 = bool(GPUS) and "GTX 16" not in GPU_NAME
TRAIN_GPUS = max(1, min(setting("TRAIN_GPUS", len(GPUS)), len(GPUS)))
ENCODE_BATCH = 1024 if USE_FP16 else 256
ENCODE_CHUNK = 1_000_000
SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32
RESCORE_EXTRA = 32
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory if GPUS else 0
print(f"sentence-transformers {sentence_transformers.__version__} | {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16} | "
      f"training on {TRAIN_GPUS} GPU(s){' | QUICK RUN' if QUICK_RUN else ''}")
print(f"model: {LAYERS} layers x {HIDDEN}, {HEADS} heads, vocab {VOCAB_SIZE:,}, max {MAX_LEN} tokens | test text: {USE_TEST_TEXT}")
print(f"stage 1: {MLM_TEXTS_PER_COUNTRY:,} texts per country, {MLM_EPOCHS:g} epoch(s), batch {MLM_BATCH}, lr {MLM_LR:g} | "
      f"stage 2: {PAIRS_PER_COUNTRY:,} matches + {NOISY_PAIRS_PER_COUNTRY:,} noisy per country, {SCRIPT_PAIRS:,} script, "
      f"{CL_EPOCHS:g} epoch(s), batch {CL_BATCH}, lr {CL_LR:g}")
if len(GPUS) < 2:
    print("WARNING: fewer than 2 GPUs: choose the accelerator 'GPU T4 x2' in the notebook settings")

## Inputs
`processed/` and the ground truth are found as in 02 full. 02 full's train buckets (for hard negatives) and word map
are found anywhere under the inputs; `BUCKET_DIR` and `WORDMAP_FILE` override the search.

In [ ]:
def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: attach 01's output (FULL_RUN=true) or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input or the local dataset folder; None if absent."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", str(Path.cwd().parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


def search_inputs(pattern):
    """Files matching pattern under /kaggle/input and /kaggle/working, base-model outputs first, then shortest path."""
    hits = [h for root in ("/kaggle/input", "/kaggle/working") if os.path.isdir(root)
            for h in glob.glob(os.path.join(root, pattern), recursive=True) if "__MACOSX" not in h]
    return sorted(set(hits), key=lambda h: ("multilingual-e5-small" not in h, len(h)))


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
assert GT_FILE is not None, "train_ground_truth.tsv not found: attach the competition data or set DATA_ROOT"
if os.environ.get("BUCKET_DIR"):
    BUCKET_DIR = Path(os.environ["BUCKET_DIR"])
else:
    _hits = [h for h in search_inputs("**/buckets/train_*.parquet") if "e5-small-er" not in h and MODEL_NAME not in h]
    BUCKET_DIR = Path(_hits[0]).parent if _hits else None
if os.environ.get("WORDMAP_FILE"):
    WORDMAP_FILE = Path(os.environ["WORDMAP_FILE"])
else:
    _hits = search_inputs("**/translit_wordmap.json")
    WORDMAP_FILE = Path(_hits[0]) if _hits else None
FILES = {(m.group(1), m.group(2), m.group(3)): f for f in sorted(PROCESSED_DIR.glob("*_s*_*.parquet"))
         if (m := re.match(r"^(train|test)_(s[123])_(.+)\.parquet$", f.name))}
TRAIN_COUNTRIES = sorted({c for sp, s, c in FILES if sp == "train" and s == "s1"})
SPLITS = ("train", "test") if USE_TEST_TEXT else ("train",)
COUNTRIES = sorted({c for sp, s, c in FILES if sp in SPLITS})
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\nbuckets (hard negatives): {BUCKET_DIR}\n"
      f"word map: {WORDMAP_FILE or 'not found, learned below'}\ncountries with labels: {TRAIN_COUNTRIES} | "
      f"all countries: {COUNTRIES}\noutput: {OUT_DIR}")
assert set(EVAL_COUNTRIES) <= set(TRAIN_COUNTRIES), f"no train data for {set(EVAL_COUNTRIES) - set(TRAIN_COUNTRIES)}"
if USE_TEST_TEXT and not any(sp == "test" for sp, _, _ in FILES):
    print("WARNING: no test files in processed/: the model will see no French (run 01 with FULL_RUN=true)")
if BUCKET_DIR is None:
    print("WARNING: no 02 full train buckets found: `matches` pairs get random negatives instead of lookalikes")


def country_files(country, sources, splits=("train",)):
    return [FILES[(sp, s, country)] for sp in splits for s in sources if (sp, s, country) in FILES]

## Text given to the model (copied from 02 full)
The functions below are copied unchanged from `02_full_e5_buckets.ipynb` (`TEXT_MODE=fixed`), as in 02b. If they
change there, change them here too: the model must be trained on the text it will embed.
* Latin-script records: `name_norm | addr_norm` from 01.
* Names or addresses with an Indian script: transliterated from the original text with anyascii and two fixes
  (Malayalam "റ്റ" → "tt", nasal dot → "n" before most consonants). Names also go through the word map
  ("praivet" → private).

New here: `native_text`, the same record with its Indian-script name or address in the **original** script
(`सन एनर्जी लिमिटेड | ...`). It feeds the tokenizer, stage 1 and the `script` pairs.

In [ ]:
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
_INDIC_BLOCKS = range(0x0900, 0x0D80, 0x80)                     # Devanagari, Bengali, ..., Malayalam (same layout)
_ANUSVARA = "".join(chr(b + 0x02) for b in _INDIC_BLOCKS)
_LABIALS = "".join(chr(b + o) for b in _INDIC_BLOCKS for o in range(0x2A, 0x2F))      # pa, pha, ba, bha, ma
RE_ANUSVARA_N = re.compile(f"[{_ANUSVARA}](?=[ऀ-ൿ])(?![{_LABIALS}])")


def translit(s):
    """anyascii with the two fixes above."""
    return anyascii(RE_ANUSVARA_N.sub("n", s.replace("റ്റ", "ട്ട")))


def latin(raw, cleaned):
    """cleaned (01's text) for Latin-script text, else the transliterated original."""
    return " ".join(translit(raw).lower().split()) if RE_NATIVE.search(raw) else cleaned


WORDMAP_MIN_PAIRS, WORDMAP_MIN_SHARE, WORDMAP_MIN_RATIO = 20, 0.6, 55
_RE_WORD2 = re.compile(r"[a-z]{2,}")


def learn_word_map(processed_dir, gt_file):
    """{transliterated word: English word} from the train true pairs of every country (only Indian-script names count)."""
    gt = pd.read_csv(gt_file, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    gp = gt[gt["matched_entity_ids"] != ""]
    gp = gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
    owner = dict(zip(gp["cand"], gp["source1_entity_id"]))
    del gt, gp
    s1_name = {}
    for f in sorted(Path(processed_dir).glob("train_s1_*.parquet")):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        s1_name.update(zip(d["entity_id"], d["business_name"].fillna("").astype(str)))
    pair_counts, tok_counts = collections.Counter(), collections.Counter()
    for f in sorted(list(Path(processed_dir).glob("train_s2_*.parquet")) + list(Path(processed_dir).glob("train_s3_*.parquet"))):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        for e, name in zip(d["entity_id"], d["business_name"].fillna("").astype(str)):
            s = owner.get(e)
            if s is None or s not in s1_name or not RE_NATIVE.search(name):
                continue
            a = set(_RE_WORD2.findall(s1_name[s].lower()))
            for t in set(_RE_WORD2.findall(translit(name).lower())):
                tok_counts[t] += 1
                if t in a or not a:
                    continue
                best = rf_process.extractOne(t, a, scorer=fuzz.ratio, score_cutoff=WORDMAP_MIN_RATIO)
                if best:
                    pair_counts[(t, best[0])] += 1
    best_of = {}
    for (t, e), n in pair_counts.items():
        if n > best_of.get(t, (0, ""))[0] or (n == best_of[t][0] and e < best_of[t][1]):
            best_of[t] = (n, e)
    return {t: e for t, (n, e) in sorted(best_of.items())
            if n >= WORDMAP_MIN_PAIRS and n / tok_counts[t] >= WORDMAP_MIN_SHARE}


if WORDMAP_FILE is not None:
    WORDMAP = json.loads(WORDMAP_FILE.read_text(encoding="utf-8"))
else:
    t0 = time.time()
    WORDMAP = learn_word_map(PROCESSED_DIR, GT_FILE)
    print(f"word map learned from the train pairs in {time.time() - t0:.0f}s")
(OUT_DIR / "translit_wordmap.json").write_text(json.dumps(WORDMAP, ensure_ascii=False, indent=0, sort_keys=True),
                                               encoding="utf-8")
print(f"word map: {len(WORDMAP)} words, e.g. " + ", ".join(f"{k} -> {v}" for k, v in list(WORDMAP.items())[:12]))


def translit_name(s):
    """Transliterated Indian-script name with the word map applied ('प्राइवेट लिमिटेड' -> 'private limited')."""
    return " ".join(WORDMAP.get(w, w) for w in translit(s).lower().split())


def latin_name(raw, cleaned):
    """latin() for names: Indian-script names transliterated and word-mapped."""
    return translit_name(raw) if RE_NATIVE.search(raw) else cleaned


def native_text(raw_name, raw_addr, name_norm, addr_norm):
    """The record with its Indian-script name or address in the original script ('' if it has none)."""
    nat_n, nat_a = RE_NATIVE.search(raw_name) is not None, RE_NATIVE.search(raw_addr) is not None
    if not (nat_n or nat_a):
        return ""
    n = " ".join(raw_name.lower().split()) if nat_n else name_norm.strip()
    a = " ".join(raw_addr.lower().split()) if nat_a else addr_norm.strip()
    return f"{n} | {a}" if a else n


TEXT_COLS = ["name_norm", "addr_norm", "business_name", "business_address"]
QUICK_KEEP = collections.defaultdict(set)        # quick run: true matches of the sampled S1, always read


def read_texts(paths, keep=frozenset()):
    """entity_ids, 02 full texts (TEXT_MODE=fixed), per record: Indian-script name, empty address, and the
    original-script text ('' for Latin-script records). QUICK_RUN: the first QUICK_ROWS rows of each file, plus `keep`."""
    if not paths:                                # e.g. France has no train files
        return np.array([], dtype=object), [], np.zeros(0, bool), np.zeros(0, bool), []
    frames = []
    for p in paths:
        d = pd.read_parquet(p, columns=["entity_id"] + TEXT_COLS)
        if QUICK_ROWS:
            d = d[(np.arange(len(d)) < QUICK_ROWS) | d["entity_id"].astype(str).isin(keep).to_numpy()]
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    for c in TEXT_COLS:
        df[c] = df[c].fillna("").astype(str)
    names = [latin_name(r, c) for r, c in zip(df["business_name"], df["name_norm"])]
    addrs = [latin(r, c) for r, c in zip(df["business_address"], df["addr_norm"])]
    texts = [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]
    native = np.fromiter((RE_NATIVE.search(x) is not None for x in df["business_name"]), bool, len(df))
    empty = np.fromiter((not a.strip() for a in addrs), bool, len(df))
    natives = [native_text(rn, ra, cn, ca) for rn, ra, cn, ca in
               zip(df["business_name"], df["business_address"], df["name_norm"], df["addr_norm"])]
    return df["entity_id"].astype(str).to_numpy(dtype=object), texts, native, empty, natives

## True pairs, and the S1 kept away from training
As in 02b: 03 full draws its LightGBM training S1 with `rng = np.random.default_rng(42)` and, for each train
country in sorted order, `rng.choice(s1_all, 200000, replace=False)` over the S1 in 01's file order. The same draw
is repeated here (checked against 02 full's bucket order when attached). Only S1 **outside** that set give
`matches` and `script_matches` pairs. The evaluation's held-out S1 come from inside it.

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
gp = gt[gt["matched_entity_ids"] != ""]
PAIRS = (gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
         .rename(columns={"source1_entity_id": "s1"})[["s1", "cand"]].reset_index(drop=True))
OWNER = pd.Series(PAIRS["s1"].to_numpy(), index=PAIRS["cand"].to_numpy())      # S2/S3 record -> its true S1
OWNER = OWNER[~OWNER.index.duplicated()]         # one owner per record (EDA: none is reused); guards .map below
del gt, gp
print(f"{len(PAIRS):,} true (S1, S2/S3) pairs in the ground truth")

S1_ORDER = {c: pd.read_parquet(FILES[("train", "s1", c)], columns=["entity_id"])["entity_id"].astype(str)
            .to_numpy(dtype=object) for c in TRAIN_COUNTRIES}
for c in TRAIN_COUNTRIES:
    path = BUCKET_DIR / f"train_{c}.parquet" if BUCKET_DIR else None
    if path is not None and path.exists():
        in_buckets = pd.read_parquet(path, columns=["s1_entity_id"])["s1_entity_id"].unique()
        assert len(in_buckets) == len(S1_ORDER[c]) and (in_buckets == S1_ORDER[c]).all(), \
            f"{c}: the S1 order of 02 full's buckets differs from processed/: 03's LightGBM S1 cannot be reproduced"
        print(f"{c}: S1 order of 02 full's train buckets matches processed/ ({len(in_buckets):,} S1)")
    if QUICK_ROWS:                               # quick run: keep the true matches of the S1 that are read
        QUICK_KEEP[c] = set(PAIRS.loc[PAIRS["s1"].isin(set(S1_ORDER[c][:QUICK_ROWS])), "cand"])

_rng03 = np.random.default_rng(LGBM_SEED)
LGBM_S1 = {c: set(_rng03.choice(S1_ORDER[c], min(LGBM_TRAIN_S1_PER_COUNTRY, len(S1_ORDER[c])), replace=False))
           for c in TRAIN_COUNTRIES}
(OUT_DIR / "lgbm_train_s1.txt").write_text("\n".join(sorted(s for v in LGBM_S1.values() for s in v)), encoding="utf-8")
pd.DataFrame([{"country": c, "train S1": len(S1_ORDER[c]), "LightGBM S1 (never trained on)": len(LGBM_S1[c]),
               "true pairs": int(PAIRS["s1"].isin(set(S1_ORDER[c])).sum())} for c in TRAIN_COUNTRIES]).set_index("country")

## Noisy copies
A record's text with 1 to 3 kinds of noise, drawn from the kinds the EDA found between true pairs. The noise hits
the cleaned text 02 full embeds, so junk wrappers and accents, which 01 already removes, are left out.

| Noise | Example |
|---|---|
| typo in a name or address word (swap, drop, double, replace a letter) | westgrove → wcstgrove |
| abbreviation or expansion | road ↔ rd, private ↔ pvt, rue ↔ r, saint ↔ st |
| legal form removed | sun energy pvt ltd → sun energy |
| address word dropped, last words (city, PIN, state) dropped, words moved | ..., pune 411001 → ... |
| house-number format | 0058 → 58, 12 → no 12 |
| duplicated or swapped name words | sun energy → sun sun energy |
| name written as a website, or address removed | sun energy pvt ltd → sunenergy.com |

France gets only these pairs, so this list is what teaches the model which differences do **not** change the
business. It is a guess at the real noise, not the real noise. That is why the France score can only be checked
through the full pipeline.

In [ ]:
ABBREVIATIONS = {
    "road": "rd", "street": "st", "avenue": "ave", "boulevard": "blvd", "drive": "dr", "lane": "ln", "court": "ct",
    "highway": "hwy", "suite": "ste", "apartment": "apt", "building": "bldg", "floor": "flr", "north": "n",
    "south": "s", "east": "e", "west": "w", "near": "nr", "opposite": "opp", "sector": "sec", "nagar": "ngr",
    "colony": "col", "private": "pvt", "limited": "ltd", "company": "co", "corporation": "corp",
    "incorporated": "inc", "international": "intl", "industries": "inds", "services": "svcs",
    "technologies": "tech", "number": "no", "centre": "ctr", "center": "ctr", "rue": "r", "saint": "st",
    "chemin": "ch", "route": "rte", "impasse": "imp", "place": "pl", "square": "sq", "mount": "mt",
}
EXPANSIONS = {}
for _full, _short in ABBREVIATIONS.items():
    EXPANSIONS.setdefault(_short, _full)
LEGAL_WORDS = {"pvt", "ltd", "private", "limited", "llp", "llc", "inc", "corp", "co", "sarl", "sas", "sasu", "sa",
               "eurl", "sci", "ei", "cie", "plc", "lp", "gmbh"}
TLDS = [".com", ".in", ".co.in", ".fr", ".net", ".org"]
LETTERS = "abcdefghijklmnopqrstuvwxyz"


def typo(word, r):
    """One keyboard-style error: swap two letters, or drop, double or replace one."""
    if len(word) < 4 or not word.isalpha():
        return word
    i, kind = r.randrange(1, len(word) - 1), r.randrange(4)
    if kind == 0:
        return word[:i] + word[i + 1] + word[i] + word[i + 2:]
    if kind == 1:
        return word[:i] + word[i + 1:]
    if kind == 2:
        return word[:i] + word[i] + word[i:]
    return word[:i] + r.choice(LETTERS) + word[i + 1:]


def _typo_in(words, r):
    idx = [i for i, w in enumerate(words) if len(w) >= 4 and w.isalpha()]
    if idx:
        i = r.choice(idx)
        words[i] = typo(words[i], r)
    return words


def op_name_typo(n, a, r):
    return _typo_in(n, r), a


def op_addr_typo(n, a, r):
    return n, _typo_in(a, r)


def op_abbreviate(n, a, r):
    spots = [(k, i) for k, ws in ((0, n), (1, a)) for i, w in enumerate(ws) if w in ABBREVIATIONS or w in EXPANSIONS]
    if spots:
        k, i = r.choice(spots)
        ws = (n, a)[k]
        ws[i] = ABBREVIATIONS.get(ws[i]) or EXPANSIONS[ws[i]]
    return n, a


def op_drop_legal(n, a, r):
    kept = [w for w in n if w.strip(".") not in LEGAL_WORDS]
    return (kept if kept else n), a


def op_drop_addr_word(n, a, r):
    if len(a) >= 3:
        del a[r.randrange(len(a))]
    return n, a


def op_drop_addr_tail(n, a, r):
    if len(a) >= 4:
        del a[-r.choice((1, 2)):]
    return n, a


def op_move_addr_words(n, a, r):
    if len(a) >= 3:
        k = r.randrange(1, len(a))
        a = a[k:] + a[:k]
    return n, a


def op_numbers(n, a, r):
    idx = [i for i, w in enumerate(a) if w.isdigit()]
    if idx:
        i = r.choice(idx)
        stripped = a[i].lstrip("0") or "0"
        if stripped != a[i]:
            a[i] = stripped
        elif i == 0 or a[i - 1] != "no":
            a.insert(i, "no")
    return n, a


def op_duplicate_word(n, a, r):
    if n:
        i = r.randrange(len(n))
        n.insert(i, n[i])
    return n, a


def op_swap_words(n, a, r):
    if len(n) >= 2:
        i = r.randrange(len(n) - 1)
        n[i], n[i + 1] = n[i + 1], n[i]
    return n, a


def op_website(n, a, r):
    core = "".join(ch for w in n if w.strip(".") not in LEGAL_WORDS for ch in w if ch.isalnum())
    return ([core + r.choice(TLDS)] if len(core) >= 3 else n), a


def op_no_address(n, a, r):
    return n, []


# weights by repetition: the common kinds of noise first
NOISE_OPS = ([op_name_typo] * 3 + [op_addr_typo] * 3 + [op_abbreviate] * 3 + [op_drop_legal] * 2
             + [op_drop_addr_word] * 2 + [op_drop_addr_tail] * 2 + [op_move_addr_words] * 2 + [op_numbers] * 2
             + [op_duplicate_word, op_swap_words, op_website, op_no_address])


def noisy_copy(text, r):
    """The text with 1-3 kinds of noise; a typo is forced if the draw left it unchanged."""
    name, _, addr = text.partition(" | ")
    out = text
    for _ in range(3):
        n, a = name.split(), addr.split()
        for op in r.sample(NOISE_OPS, r.choice((1, 1, 2, 2, 3))):
            n, a = op(n, a, r)
        out = " ".join(n) + (" | " + " ".join(a) if a else "")
        if out != text and n:
            return out
    n, a = _typo_in(name.split(), r), addr.split()
    return " ".join(n) + (" | " + " ".join(a) if a else "")


_r = random.Random(0)
for _t in ["sun energy pvt ltd | 12 mg road near city mall pune maharashtra 411001",
           "boulangerie du centre sarl | 0058 rue saint michel bordeaux 33000",
           "rippey beyond inc | 1301 lebanon pike nashville tn 37210"]:
    print(f"{_t}\n  -> " + "\n  -> ".join(noisy_copy(_t, _r) for _ in range(3)))

## Corpus and training pairs
Country by country:
* **corpus:** every distinct text (02 full's) and every original-script text, from train and (with `USE_TEST_TEXT`)
  test records. It feeds the tokenizer and stage 1.
* **`noisy`:** `NOISY_PAIRS_PER_COUNTRY` random records of the country (train and test) with a noisy copy.
* **`script`:** up to `SCRIPT_PAIRS` Indian-script records: 02 full's transliterated text ↔ original script.
* **`matches`** (countries with labels): up to `PAIRS_PER_COUNTRY` random true pairs whose S1 is outside LightGBM's
  set, `(S1 text, true record's text, lookalike's text)`. The lookalike is drawn among the S1's `HARD_NEG_TOP`
  best-scored bucket candidates that are not its matches, as in 02b.
* **`script_matches`:** those pairs whose true record is written in an Indian script, with the record in its
  original script.

Train and test are read separately, because their entity IDs may overlap. The files go to `WORK_DIR`, which is
not part of the notebook's output.

**Leak check**, before any training:
* The notebook stops if a `matches` pair involves one of LightGBM's S1 (the evaluation's "unseen" S1) or one of their
  true records. `pairs touching evaluation S1` must be 0.
* `evaluation pairs with a trained pair's exact texts` counts evaluation pairs whose two texts are exactly those of
  a training pair (other entities with identical text). Expect 0 or a handful.
* The corpus, `noisy` and `script` rows include the evaluation records' own text **by design**. They carry no
  pairing: nothing tells the model which record matches which S1. The model gets exactly the same treatment for
  the test records, so this mirrors test conditions rather than inflating the score.

In [ ]:
def hard_negative_lists(country, s1_ids):
    """{S1: its HARD_NEG_TOP best-scored bucket candidates that are not its true matches}, from 02 full's train buckets."""
    path = BUCKET_DIR / f"train_{country}.parquet" if BUCKET_DIR else None
    if path is None or not path.exists():
        return {}
    cols = [c for c in ("s1_entity_id", "candidate_entity_id", "score") if c in pq.read_schema(path).names]
    t = pq.read_table(path, columns=cols)
    keep = pc.is_in(t["s1_entity_id"].cast(pa.large_string()), value_set=pa.array(sorted(s1_ids), pa.large_string()))
    b = t.filter(keep).to_pandas()
    del t, keep
    b = b[b["candidate_entity_id"].map(OWNER).to_numpy() != b["s1_entity_id"].to_numpy()]   # drop the true matches
    if "score" in b:
        b = b.sort_values(["s1_entity_id", "score"], ascending=[True, False], kind="stable")
    b = b.groupby("s1_entity_id", sort=False).head(HARD_NEG_TOP)
    return {s: g.to_numpy(dtype=object) for s, g in b.groupby("s1_entity_id", sort=False)["candidate_entity_id"]}


def save_pairs(name, anchors, positives, negatives=None):
    """Write a pair file (texts prefixed as in 02 full) and register it for stage 2."""
    if not len(anchors):
        return 0
    frame = pd.DataFrame({"anchor": [PREFIX + t for t in anchors], "positive": [PREFIX + t for t in positives]})
    if negatives is not None:
        frame["negative"] = [PREFIX + t for t in negatives]
    frame = frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    frame.to_parquet(PAIR_DIR / f"{name}.parquet", index=False)
    PAIR_FILES[name] = str(PAIR_DIR / f"{name}.parquet")
    EXAMPLES.extend((name, *row) for row in frame.head(2).itertuples(index=False))
    return len(frame)


rng = np.random.default_rng(SEED)
prng = random.Random(SEED)
PAIR_FILES, USED_PAIRS, STATS, EXAMPLES = {}, {}, [], []
for country in COUNTRIES:
    t0 = time.time()
    tr = read_texts(country_files(country, ("s1", "s2", "s3"), ("train",)), keep=QUICK_KEEP.get(country, frozenset()))
    te = read_texts(country_files(country, ("s1", "s2", "s3"), ("test",))) if USE_TEST_TEXT else None
    texts = tr[1] + (te[1] if te else [])
    natives = tr[4] + (te[4] if te else [])
    row = {"country": country, "records": len(texts)}

    corpus = pd.unique(np.array([t for t in itertools.chain(texts, natives) if t], dtype=object))
    pd.DataFrame({"text": [PREFIX + t for t in corpus]}).to_parquet(CORPUS_DIR / f"{country}.parquet", index=False)
    row["corpus texts"] = len(corpus)
    del corpus

    filled = np.flatnonzero(np.fromiter((bool(t) for t in texts), bool, len(texts)))
    pick = rng.choice(filled, min(NOISY_PAIRS_PER_COUNTRY, len(filled)), replace=False)
    row["noisy"] = save_pairs(f"noisy_{country}", [texts[i] for i in pick], [noisy_copy(texts[i], prng) for i in pick])

    both = np.flatnonzero(np.fromiter((bool(n) and bool(t) and n != t for t, n in zip(texts, natives)), bool, len(texts)))
    pick = rng.choice(both, min(SCRIPT_PAIRS, len(both)), replace=False)
    row["script"] = save_pairs(f"script_{country}", [texts[i] for i in pick], [natives[i] for i in pick])
    del texts, natives, te

    if country in TRAIN_COUNTRIES:
        ids, tr_texts, _, _, tr_natives = tr
        text_of, native_of = dict(zip(ids, tr_texts)), dict(zip(ids, tr_natives))
        pool = np.array([i for i in ids if not i.startswith("S1-")], dtype=object)       # the country's S2/S3 records
        p = PAIRS[PAIRS["s1"].isin(set(S1_ORDER[country]) - LGBM_S1[country])]
        ok = p["s1"].map(text_of).fillna("").astype(bool) & p["cand"].map(text_of).fillna("").astype(bool)
        p = p[ok.to_numpy()]
        if len(p) > PAIRS_PER_COUNTRY:
            p = p.iloc[np.sort(rng.choice(len(p), PAIRS_PER_COUNTRY, replace=False))]
        p = p.iloc[rng.permutation(len(p))].reset_index(drop=True)
        anc = p["s1"].map(text_of).to_numpy(dtype=object)
        pos = p["cand"].map(text_of).to_numpy(dtype=object)
        lists = hard_negative_lists(country, set(p["s1"]))
        neg, from_bucket = np.empty(len(p), dtype=object), 0
        for i, s in enumerate(p["s1"].to_numpy()):
            opts, chosen = lists.get(s), None
            if opts is not None and len(opts):
                for _ in range(3):
                    t = text_of.get(opts[rng.integers(len(opts))], "")
                    if t and t != anc[i] and t != pos[i]:
                        chosen, from_bucket = t, from_bucket + 1
                        break
            while chosen is None:
                t = text_of.get(pool[rng.integers(len(pool))], "")
                if t and t != anc[i] and t != pos[i]:
                    chosen = t
            neg[i] = chosen
        # leak check, before any training: no evaluation S1 (LightGBM's) and none of their true records in these pairs
        # (script_matches are a subset of them)
        ev = PAIRS[PAIRS["s1"].isin(LGBM_S1[country])]
        row["pairs touching evaluation S1"] = int(p["s1"].isin(LGBM_S1[country]).sum() + p["cand"].isin(set(ev["cand"])).sum())
        assert row["pairs touching evaluation S1"] == 0, \
            f"{country}: {row['pairs touching evaluation S1']} training pairs involve an evaluation S1 or its true records"
        trained_texts = set(zip(anc, pos))
        row["evaluation pairs with a trained pair's exact texts"] = sum(
            (a, b) in trained_texts for a, b in zip(ev["s1"].map(text_of).fillna(""), ev["cand"].map(text_of).fillna("")))
        del ev, trained_texts
        row["matches"] = save_pairs(f"matches_{country}", anc, pos, neg)
        row["hard negative from the bucket"] = round(from_bucket / max(len(p), 1), 3)
        USED_PAIRS[country] = set(p["s1"] + "|" + p["cand"])
        nat = p["cand"].map(native_of).fillna("").to_numpy(dtype=object)
        with_native = np.flatnonzero(nat != "")[:SCRIPT_PAIRS]
        row["script_matches"] = save_pairs(f"script_matches_{country}", anc[with_native], nat[with_native])
        del text_of, native_of, pool, lists, anc, pos, neg, nat
    del tr
    gc.collect()
    row["seconds"] = round(time.time() - t0)
    STATS.append(row)
    print(f"{country}: {row}", flush=True)

(OUT_DIR / "matches_s1.txt").write_text("\n".join(sorted({k.split("|")[0] for v in USED_PAIRS.values() for k in v})),
                                        encoding="utf-8")
for name, *texts_ in EXAMPLES:
    print(f"{name}:\n    " + "\n    ".join(t[len(PREFIX):][:110] for t in texts_))
pd.DataFrame(STATS).set_index("country").fillna(0)

## Tokenizer
WordPiece on up to `TOKENIZER_TEXTS` corpus texts, the same number from each country: NFKC and lower case, split on
spaces and punctuation, 32,000 tokens. The alphabet keeps up to 3,000 characters, so the Indian scripts and French
letters all get tokens. Skipped when `MLM_MODEL` or `TRAINED_MODEL` is set: that model brings its own tokenizer.

In [ ]:
SPECIAL = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]


def sample_corpus(per_country, seed):
    """Up to per_country random corpus texts from each country, shuffled."""
    r = np.random.default_rng(seed)
    out = []
    for country in COUNTRIES:
        texts = pq.read_table(CORPUS_DIR / f"{country}.parquet").column("text").to_pylist()
        if len(texts) > per_country:
            texts = [texts[i] for i in r.choice(len(texts), per_country, replace=False)]
        out += texts
    return [out[i] for i in r.permutation(len(out))]


if not (MLM_MODEL or TRAINED_MODEL):
    t0 = time.time()
    sample = sample_corpus(max(1, TOKENIZER_TEXTS // len(COUNTRIES)), SEED)
    tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tok.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
    tok.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    tok.decoder = decoders.WordPiece()
    tok.train_from_iterator(sample, trainer=trainers.WordPieceTrainer(
        vocab_size=VOCAB_SIZE, min_frequency=5, special_tokens=SPECIAL, limit_alphabet=3000,
        continuing_subword_prefix="##", show_progress=False), length=len(sample))
    assert [tok.token_to_id(t) for t in SPECIAL] == [0, 1, 2, 3, 4], "special tokens must be ids 0-4"
    tok.post_processor = processors.TemplateProcessing(single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
                                                       special_tokens=[("[CLS]", 2), ("[SEP]", 3)])
    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    tok.save(str(TOKENIZER_DIR / "raw_tokenizer.json"))
    PreTrainedTokenizerFast(tokenizer_object=tok, model_max_length=MAX_LEN, pad_token="[PAD]", unk_token="[UNK]",
                            cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]").save_pretrained(str(TOKENIZER_DIR))
    VOCAB = tok.get_vocab_size()
    print(f"tokenizer: {VOCAB:,} tokens from {len(sample):,} texts in {time.time() - t0:.0f}s")
    for t in sample[:3] + [s for s in sample if RE_NATIVE.search(s)][:2]:
        print(f"  {t[:80]}\n    -> {tok.encode(t).tokens[:24]}")
    del sample
else:
    print(f"tokenizer: from {MLM_MODEL or TRAINED_MODEL}")

## Stage 1: masked-language-model pretraining
`MLM_TEXTS_PER_COUNTRY` random corpus texts per country are tokenized once into a flat token file. Two processes
(one per GPU, `torchrun`) train the BERT encoder:
* 15% of the tokens are hidden: 80% become `[MASK]`, 10% a random token, 10% stay. The model predicts them, through
  a head tied to its input embeddings, scored only at the hidden positions (faster than scoring every position).
* Batches of similar length (less padding), fp16, AdamW, 6% warm-up then linear decay, gradients clipped at 1.
* The log shows the loss and the share of hidden tokens guessed right. Both should improve steadily.

The encoder and the tokenizer are saved in `er-embed-small-mlm/`, every 5,000 steps and at the end. If a later step
fails, `MLM_MODEL=<that folder>` skips this stage.

In [ ]:
MLM_SCRIPT = r'''
"""Stage 1 of 02c: masked-language-model pretraining of a small BERT (one process per GPU under torchrun)."""
import datetime
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F
from transformers import BertConfig, BertModel

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
world = int(os.environ.get("WORLD_SIZE", 1))
rank = int(os.environ.get("RANK", 0))
local = int(os.environ.get("LOCAL_RANK", 0))
cuda = torch.cuda.is_available()
if world > 1:
    dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=10))
device = torch.device(f"cuda:{local}" if cuda else "cpu")
if cuda:
    torch.cuda.set_device(device)
torch.manual_seed(cfg["seed"] + rank)
main = rank == 0
fp16 = cuda and cfg["fp16"]

tokens = np.memmap(cfg["tokens_file"], dtype=np.uint16, mode="r")
offsets = np.load(cfg["offsets_file"])
lengths = np.diff(offsets)
n = len(lengths)
pad, cls_id, sep_id, mask_id = cfg["pad_id"], cfg["cls_id"], cfg["sep_id"], cfg["mask_id"]
per_gpu = max(1, cfg["batch_size"] // world)


def epoch_batches(epoch):
    """This process's batches for one epoch: rows shuffled, sorted by length within groups of 50 batches, the batches
    shuffled again, then split evenly between the processes."""
    r = np.random.default_rng(cfg["seed"] + epoch)
    order = r.permutation(n)
    group, batches = per_gpu * 50, []
    for s in range(0, n, group):
        g = order[s:s + group]
        g = g[np.argsort(lengths[g], kind="stable")]
        batches += [g[i:i + per_gpu] for i in range(0, len(g) - per_gpu + 1, per_gpu)]
    batches = [batches[i] for i in r.permutation(len(batches))]
    steps = len(batches) // world
    return batches[rank * steps:(rank + 1) * steps]


def make_batch(rows):
    """Padded token ids of the rows, with 15% of the tokens hidden (80% [MASK], 10% random, 10% unchanged)."""
    ids = np.full((len(rows), int(lengths[rows].max())), pad, np.int64)
    for k, i in enumerate(rows):
        ids[k, :lengths[i]] = tokens[offsets[i]:offsets[i + 1]]
    ids = torch.from_numpy(ids).to(device)
    attention = ids != pad
    hidden = (torch.rand(ids.shape, device=device) < cfg["mlm_prob"]) & attention & (ids != cls_id) & (ids != sep_id)
    labels = torch.where(hidden, ids, torch.full_like(ids, -100))
    roll = torch.rand(ids.shape, device=device)
    inputs = torch.where(hidden & (roll < 0.8), torch.full_like(ids, mask_id), ids)
    inputs = torch.where(hidden & (roll >= 0.8) & (roll < 0.9),
                         torch.randint_like(ids, cfg["n_special"], cfg["vocab_size"]), inputs)
    return inputs, attention.long(), labels


config = BertConfig(vocab_size=cfg["vocab_size"], hidden_size=cfg["hidden"], num_hidden_layers=cfg["layers"],
                    num_attention_heads=cfg["heads"], intermediate_size=4 * cfg["hidden"],
                    max_position_embeddings=cfg["max_len"], type_vocab_size=2, pad_token_id=pad)


class MaskedLM(torch.nn.Module):
    """BERT encoder + prediction head tied to the input embeddings, scored only at the hidden positions."""

    def __init__(self):
        super().__init__()
        self.bert = BertModel(config, add_pooling_layer=False)
        h = config.hidden_size
        self.transform = torch.nn.Sequential(torch.nn.Linear(h, h), torch.nn.GELU(),
                                             torch.nn.LayerNorm(h, eps=config.layer_norm_eps))
        self.bias = torch.nn.Parameter(torch.zeros(config.vocab_size))

    def forward(self, input_ids, attention_mask, labels):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        picked = labels != -100
        logits = self.transform(out[picked]) @ self.bert.embeddings.word_embeddings.weight.t() + self.bias
        target = labels[picked]
        return F.cross_entropy(logits.float(), target), (logits.argmax(-1) == target).float().mean()


model = MaskedLM().to(device)
core = model
if world > 1:
    model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local] if cuda else None)
params = list(core.named_parameters())
opt = torch.optim.AdamW([{"params": [p for _, p in params if p.ndim > 1], "weight_decay": 0.01},
                         {"params": [p for _, p in params if p.ndim <= 1], "weight_decay": 0.0}],
                        lr=cfg["lr"], betas=(0.9, 0.98), eps=1e-6)
steps_per_epoch = len(epoch_batches(0))
total = max(1, int(steps_per_epoch * cfg["epochs"]))
warmup = max(1, int(total * cfg["warmup_share"]))
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min((s + 1) / warmup, max(0.0, (total - s) / max(1, total - warmup))))
try:
    scaler = torch.amp.GradScaler("cuda", enabled=fp16)
except (AttributeError, TypeError):                  # PyTorch < 2.3
    scaler = torch.cuda.amp.GradScaler(enabled=fp16)
if main:
    print(f"{world} process(es) x {per_gpu} texts per step | {n:,} texts, {total:,} steps "
          f"({cfg['epochs']:g} epoch(s)) | {sum(p.numel() for _, p in params) / 1e6:.1f}M parameters", flush=True)


def save():
    core.bert.save_pretrained(cfg["out_dir"])


log, step, epoch, t0 = [], 0, 0, time.time()
sum_loss = sum_acc = torch.zeros((), device=device)
t_last, last_step = time.time(), 0
while step < total:
    for rows in epoch_batches(epoch):
        if step >= total:
            break
        inputs, attention, labels = make_batch(rows)
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            loss, acc = model(inputs, attention, labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        sum_loss, sum_acc = sum_loss + loss.detach(), sum_acc + acc.detach()
        if step % cfg["log_every"] == 0 or step == total:
            k = step - last_step
            entry = {"step": step, "loss": round(sum_loss.item() / k, 4), "masked accuracy": round(sum_acc.item() / k, 4),
                     "lr": sched.get_last_lr()[0]}
            if main:
                minutes = (time.time() - t0) / 60
                speed = k * per_gpu * world / max(time.time() - t_last, 1e-9)
                print(f"step {step:,}/{total:,}  loss {entry['loss']:.4f}  masked accuracy {entry['masked accuracy']:.3f}  "
                      f"lr {entry['lr']:.2e}  {speed:,.0f} texts/s  {minutes:.1f} min, "
                      f"~{minutes / step * (total - step):.0f} min left", flush=True)
                log.append(entry)
            sum_loss = sum_acc = torch.zeros((), device=device)
            t_last, last_step = time.time(), step
        if main and step % cfg["save_every"] == 0:
            save()
    epoch += 1

if main:
    save()
    pd.DataFrame(log).to_csv(cfg["log_file"], index=False)
    Path(cfg["done_file"]).write_text(json.dumps({"minutes": round((time.time() - t0) / 60, 1), "processes": world,
                                                  "steps": total, "texts": n}), encoding="utf-8")
if world > 1:
    dist.barrier()                                   # the other process waits until the model is saved
    dist.destroy_process_group()
'''


def launch(label, script, run, n_gpus):
    """Run a training script with n_gpus processes (torchrun when more than one), streaming its output; exit code."""
    path, run_file = OUT_DIR / f"{label}.py", WORK_DIR / f"{label}_run.json"
    path.write_text(script, encoding="utf-8")
    run_file.write_text(json.dumps(run), encoding="utf-8")
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    env.setdefault("NCCL_P2P_DISABLE", "1")      # GPU-to-GPU traffic through the host: direct copies hang on some cloud machines
    if n_gpus > 1:
        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={n_gpus}",
               str(path), str(run_file)]
    else:
        cmd = [sys.executable, str(path), str(run_file)]
        if GPUS:                                 # one visible GPU, or the Trainer would wrap the model in DataParallel
            env["CUDA_VISIBLE_DEVICES"] = env.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
                            encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def train_stage(label, script, run):
    """launch() on TRAIN_GPUS GPUs; if that fails with several, once more on one GPU. Returns the run summary."""
    done = Path(run["done_file"])
    done.unlink(missing_ok=True)
    code = launch(label, script, run, TRAIN_GPUS)
    if code != 0 and TRAIN_GPUS > 1:
        print(f"\n{label} on {TRAIN_GPUS} GPUs failed (exit code {code}): once more on one GPU\n", flush=True)
        code = launch(label, script, run, 1)
    assert code == 0 and done.exists(), f"{label} failed (exit code {code}): see the output above"
    return json.loads(done.read_text(encoding="utf-8"))


MLM_SUMMARY = {}
if not (MLM_MODEL or TRAINED_MODEL):
    t0 = time.time()
    raw = Tokenizer.from_file(str(TOKENIZER_DIR / "raw_tokenizer.json"))
    raw.enable_truncation(max_length=MAX_LEN)
    raw.no_padding()
    VOCAB = raw.get_vocab_size()
    assert VOCAB < 2 ** 16, "token ids are stored as uint16"
    texts = sample_corpus(MLM_TEXTS_PER_COUNTRY, SEED + 1)
    tokens_file, offsets_file = WORK_DIR / "mlm_tokens.bin", WORK_DIR / "mlm_offsets.npy"
    lens = []
    with open(tokens_file, "wb") as f:
        for s in range(0, len(texts), 500_000):
            enc = raw.encode_batch(texts[s:s + 500_000])
            chunk = [e.ids for e in enc]
            lens += [len(c) for c in chunk]
            f.write(np.fromiter(itertools.chain.from_iterable(chunk), np.uint16, sum(len(c) for c in chunk)).tobytes())
    np.save(offsets_file, np.concatenate([[0], np.cumsum(lens)]).astype(np.int64))
    lens = np.array(lens)
    print(f"stage 1 data: {len(texts):,} texts, {lens.sum():,} tokens (mean {lens.mean():.1f}, 99th percentile "
          f"{np.percentile(lens, 99):.0f}, {100 * (lens >= MAX_LEN).mean():.2f}% cut at {MAX_LEN}), "
          f"{time.time() - t0:.0f}s", flush=True)
    del texts, lens
    gc.collect()
    MLM_DIR.mkdir(parents=True, exist_ok=True)
    for f in TOKENIZER_DIR.iterdir():                # the tokenizer travels with the encoder
        if f.name != "raw_tokenizer.json":
            shutil.copy(f, MLM_DIR / f.name)
    MLM_SUMMARY = train_stage("stage1_mlm", MLM_SCRIPT, {
        "tokens_file": str(tokens_file), "offsets_file": str(offsets_file), "vocab_size": VOCAB, "hidden": HIDDEN,
        "layers": LAYERS, "heads": HEADS, "max_len": MAX_LEN, "pad_id": 0, "cls_id": 2, "sep_id": 3, "mask_id": 4,
        "n_special": len(SPECIAL), "batch_size": MLM_BATCH, "lr": MLM_LR, "epochs": MLM_EPOCHS, "warmup_share": 0.06,
        "mlm_prob": MLM_PROB, "fp16": USE_FP16, "seed": SEED, "out_dir": str(MLM_DIR),
        "log_file": str(OUT_DIR / "mlm_log.csv"), "done_file": str(WORK_DIR / "mlm_done.json"),
        "log_every": 20 if QUICK_RUN else 200, "save_every": 5000})
    print(f"stage 1 done: {MLM_SUMMARY}; encoder and tokenizer in {MLM_DIR}")
    for f in (tokens_file, offsets_file):
        f.unlink(missing_ok=True)
elif MLM_MODEL:
    print(f"stage 1 skipped: MLM_MODEL={MLM_MODEL}")
MLM_LOG = pd.read_csv(OUT_DIR / "mlm_log.csv") if (OUT_DIR / "mlm_log.csv").exists() else pd.DataFrame()
MLM_LOG.iloc[np.unique(np.linspace(0, len(MLM_LOG) - 1, 12).astype(int))] if len(MLM_LOG) else "no stage-1 log"

## Stage 2: contrastive training
The stage-1 encoder with mean pooling, trained with 02b's recipe on all pair files at once:
* `MultipleNegativesRankingLoss`, cosine × `SCALE`. The embeddings are gathered across the GPUs (when this
  sentence-transformers version can), so each anchor competes with all `CL_BATCH` rows of the step. Above 256 rows
  per GPU (one-GPU fallback), the cached version computes the same loss 128 rows at a time.
* `NO_DUPLICATES`: a text never appears twice in a batch. `PROPORTIONAL`: each batch comes from one pair file, in
  proportion to its rows. The in-batch negatives are then texts of the same country and kind.
* Learning rate `CL_LR` with 5% warm-up. Higher than 02b's 2e-5, because a small model fresh from MLM has more to
  learn than e5.

The loss should fall quickly, then flatten. Skipped with `TRAINED_MODEL`, which is copied to `er-embed-small/`.

In [ ]:
CL_SCRIPT = r'''
"""Stage 2 of 02c: contrastive training (one process per GPU under torchrun, or a single process)."""
import inspect
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from transformers import TrainerCallback
from transformers.trainer_callback import PrinterCallback
try:                                             # sentence-transformers >= 6 moved these; older versions keep the old path
    from sentence_transformers.sentence_transformer import losses
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers, MultiDatasetBatchSamplers
except ImportError:
    from sentence_transformers import losses
    from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
world = int(os.environ.get("WORLD_SIZE", 1))
rank = int(os.environ.get("LOCAL_RANK", 0))
per_device = max(1, cfg["batch_size"] // world)
model = SentenceTransformer(cfg["init_model"], device=f"cuda:{rank}" if torch.cuda.is_available() else "cpu")
model.max_seq_length = cfg["max_seq_length"]
data = {name: Dataset.from_pandas(pd.read_parquet(p), preserve_index=False) for name, p in cfg["train_files"].items()}
train_data = next(iter(data.values())) if len(data) == 1 else data

cached = per_device > cfg["cache_above"]
loss_cls = losses.CachedMultipleNegativesRankingLoss if cached else losses.MultipleNegativesRankingLoss
kwargs = {"scale": cfg["scale"]}
if cached:
    kwargs["mini_batch_size"] = cfg["mini_batch"]
gather = world > 1 and "gather_across_devices" in inspect.signature(loss_cls.__init__).parameters
if gather:
    kwargs["gather_across_devices"] = True
loss = loss_cls(model, **kwargs)
steps = sum(len(d) // cfg["batch_size"] for d in data.values()) * cfg["epochs"]

args = SentenceTransformerTrainingArguments(
    output_dir=cfg["checkpoint_dir"],
    num_train_epochs=cfg["epochs"],
    per_device_train_batch_size=per_device,
    learning_rate=cfg["learning_rate"],
    warmup_steps=int(cfg["warmup_share"] * steps),
    fp16=cfg["fp16"],
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
    dataloader_drop_last=True,
    save_strategy="no",
    logging_steps=cfg["logging_steps"],
    report_to="none",
    seed=cfg["seed"],
    disable_tqdm=True,
    ddp_timeout=600,                             # a stuck GPU link fails after 10 min instead of 30
)


class Progress(TrainerCallback):
    """Step, loss and time left at every log (first process only)."""

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_world_process_zero and logs and "loss" in logs:
            minutes = (time.time() - self.t0) / 60
            left = minutes / max(state.global_step, 1) * (state.max_steps - state.global_step)
            print(f"step {state.global_step:,}/{state.max_steps:,}  loss {logs['loss']:.4f}  "
                  f"lr {logs.get('learning_rate', 0):.2e}  {minutes:.1f} min, ~{left:.0f} min left", flush=True)


trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_data, loss=loss, callbacks=[Progress()])
trainer.remove_callback(PrinterCallback)
if trainer.is_world_process_zero():
    chunks = f" in chunks of {cfg['mini_batch']} rows" if cached else ""
    negatives = " gathered across GPUs" if gather else (" per GPU only (this version cannot gather)" if world > 1 else "")
    print(f"{world} process(es) x {per_device} rows per step | {loss_cls.__name__}{chunks} | in-batch negatives"
          f"{negatives} | pair files: " + ", ".join(f"{k} {len(v):,}" for k, v in data.items()), flush=True)
t0 = time.time()
trainer.train()
if trainer.is_world_process_zero():
    model.save(cfg["model_dir"])
    log = pd.DataFrame([r for r in trainer.state.log_history if "loss" in r])
    (log if len(log) else pd.DataFrame(columns=["step", "loss", "learning_rate"])).to_csv(cfg["log_file"], index=False)
    Path(cfg["done_file"]).write_text(json.dumps({
        "minutes": round((time.time() - t0) / 60, 1), "processes": world, "rows_per_gpu": per_device,
        "loss": loss_cls.__name__, "gather_across_gpus": gather}), encoding="utf-8")
if torch.distributed.is_available() and torch.distributed.is_initialized():
    torch.distributed.barrier()                  # the other processes wait until the model is saved
    torch.distributed.destroy_process_group()
'''

if TRAINED_MODEL:
    assert (Path(TRAINED_MODEL) / "modules.json").exists(), f"TRAINED_MODEL={TRAINED_MODEL} is not a model folder"
    if Path(TRAINED_MODEL).resolve() != MODEL_DIR.resolve():
        shutil.copytree(TRAINED_MODEL, MODEL_DIR, dirs_exist_ok=True)
    CL_SUMMARY = {"reused": TRAINED_MODEL}
    print(f"training skipped: TRAINED_MODEL={TRAINED_MODEL}, copied to {MODEL_DIR}")
else:
    assert (MLM_DIR / "config.json").exists(), f"no stage-1 model in {MLM_DIR}"
    n_rows = {k: pq.ParquetFile(p).metadata.num_rows for k, p in PAIR_FILES.items()}
    TRAIN_FILES = {k: p for k, p in PAIR_FILES.items() if n_rows[k] >= 2 * CL_BATCH}   # smaller files give no full batch
    left_out = sorted(set(PAIR_FILES) - set(TRAIN_FILES))
    rows = sum(n_rows[k] for k in TRAIN_FILES)
    print(f"stage 2: {rows:,} rows in {len(TRAIN_FILES)} pair files, ~{int(rows // CL_BATCH * CL_EPOCHS):,} steps of "
          f"{CL_BATCH} rows on {TRAIN_GPUS} GPU(s)" + (f" | left out (too small): {left_out}" if left_out else ""),
          flush=True)
    CL_SUMMARY = train_stage("stage2_contrastive", CL_SCRIPT, {
        "init_model": str(MLM_DIR), "train_files": TRAIN_FILES, "batch_size": CL_BATCH, "epochs": CL_EPOCHS,
        "learning_rate": CL_LR, "warmup_share": 0.05, "scale": SCALE, "seed": SEED, "fp16": USE_FP16,
        "max_seq_length": MAX_LEN, "mini_batch": 128, "cache_above": 256,
        "checkpoint_dir": str(WORK_DIR / "checkpoints"), "model_dir": str(MODEL_DIR),
        "log_file": str(OUT_DIR / "contrastive_log.csv"), "done_file": str(WORK_DIR / "contrastive_done.json"),
        "logging_steps": 10 if QUICK_RUN else 50})
    print(f"stage 2 done: {CL_SUMMARY}; model in {MODEL_DIR}")
CL_LOG = pd.read_csv(OUT_DIR / "contrastive_log.csv") if (OUT_DIR / "contrastive_log.csv").exists() else pd.DataFrame()
CL_LOG[["step", "loss", "learning_rate"]].iloc[np.unique(np.linspace(0, len(CL_LOG) - 1, 12).astype(int))] \
    if len(CL_LOG) else "no stage-2 log"

## Evaluation at full size: our model vs e5
02b's evaluation, for every model in `COMPARE_MODELS` and ours. For each country in `EVAL_COUNTRIES`, each model
encodes **every** train S2/S3 record of the country, as 02 full does, and searches them exactly (mean-centered
vectors, 02 full's code). Two groups of S1:
* **unseen:** `EVAL_S1` S1 with at least one true match, from LightGBM's set. Our model never trained on their pairs.
  This is the number that should carry over to test.
* **seen:** `SEEN_EVAL_S1` S1 of the `matches` pairs, counting only the pairs trained on. How far this exceeds
  "unseen" shows how much the model memorized.

Recall@k is the share of true pairs whose record is among the S1's k best. It is shown for all pairs and by kind of
true record: Indian-script name, empty address, Latin script with an address. `gain` is ours minus the first model
of `COMPARE_MODELS`. The encoding speed is printed too: a faster model also makes 02 full faster.

In [ ]:
def load_encoder(name):
    """The model on the CPU when it is copied to several GPUs (as in 02 full), else on the GPU; fp16 on T4s."""
    m = SentenceTransformer(str(name), device="cpu" if len(GPUS) > 1 else (GPUS[0] if GPUS else "cpu"))
    m.max_seq_length = MAX_LEN
    if USE_FP16:
        m.half()
    return m


def embed_all(model, texts, label):
    """Unit-norm float16 embeddings aligned with texts: each unique text encoded once, longest first, on all GPUs."""
    codes, uniq = pd.factorize(np.asarray(texts, dtype=object))
    order = np.argsort(-np.fromiter((len(t) for t in uniq), np.int64, len(uniq)), kind="stable")
    pool = model.start_multi_process_pool(target_devices=GPUS) if len(GPUS) > 1 else None
    out, t0 = None, time.time()
    try:
        for s in range(0, len(order), ENCODE_CHUNK):
            idx = order[s:s + ENCODE_CHUNK]
            batch = [PREFIX + t for t in uniq[idx]]
            if pool is None:
                vecs = model.encode(batch, batch_size=ENCODE_BATCH, convert_to_numpy=True, show_progress_bar=False)
            elif "pool" in inspect.signature(model.encode).parameters:           # sentence-transformers >= 5
                vecs = model.encode(batch, pool=pool, batch_size=ENCODE_BATCH, chunk_size=16 * ENCODE_BATCH,
                                    convert_to_numpy=True, show_progress_bar=False)
            else:
                vecs = model.encode_multi_process(batch, pool, batch_size=ENCODE_BATCH, chunk_size=16 * ENCODE_BATCH)
            vecs = np.asarray(vecs, dtype=np.float32)
            vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
            if out is None:
                out = np.empty((len(uniq), vecs.shape[1]), np.float16)
            out[idx] = vecs.astype(np.float16)
        print(f"  {label}: {len(uniq):,} unique texts, {len(uniq) / max(time.time() - t0, 1e-9):,.0f} texts/s", flush=True)
    finally:
        if pool is not None:
            SentenceTransformer.stop_multi_process_pool(pool)
    return out[codes]


def _normalize(x):
    return x / x.norm(dim=1, keepdim=True).clamp_min(1e-12)


def exact_search(base16, queries16, mu, k):
    """Top-k rows of base16 for every row of queries16, by inner product of mean-centered, re-normalized vectors
    (02 full's search)."""
    n, d = base16.shape
    scores = np.empty((len(queries16), k), np.float32)
    hits = np.empty((len(queries16), k), np.int64)
    if not GPUS:
        base = base16.astype(np.float32) - mu
        base /= np.maximum(np.linalg.norm(base, axis=1, keepdims=True), 1e-12)
        for s in range(0, len(queries16), 1024):
            q = queries16[s:s + 1024].astype(np.float32) - mu
            q /= np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
            sims = q @ base.T
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            order = np.argsort(-np.take_along_axis(sims, top, axis=1), axis=1)
            hits[s:s + 1024] = np.take_along_axis(top, order, axis=1)
            scores[s:s + 1024] = np.take_along_axis(sims, hits[s:s + 1024], axis=1)
        return scores, hits
    shortlist = min(n, k + RESCORE_EXTRA)
    elem = 2 if SEARCH_DTYPE == torch.float16 else 4
    rows = int(np.clip((GPU_MEMORY // 8) // (n * elem), 64, 4096))    # query rows per block: score block <= 1/8 of GPU memory
    blocks = [(s, min(s + rows, len(queries16))) for s in range(0, len(queries16), rows)]

    def run(device, my_blocks):
        with torch.cuda.device(device):
            mu_t = torch.from_numpy(mu).to(device)
            base = torch.empty((n, d), dtype=SEARCH_DTYPE, device=device)
            step = max(100_000, 384_000_000 // d)
            for s in range(0, n, step):
                chunk = torch.from_numpy(base16[s:s + step]).to(device).float() - mu_t
                base[s:s + len(chunk)] = _normalize(chunk).to(SEARCH_DTYPE)
            del chunk
            for s, e in my_blocks:
                q = _normalize(torch.from_numpy(queries16[s:e]).to(device).float() - mu_t)
                cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                top = torch.topk(rescored, k, dim=1)
                scores[s:e] = top.values.cpu().numpy()
                hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
            del base
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(len(GPUS)) as ex:
        list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
    return scores, hits


EVAL_MODELS = [(m.rstrip("/").split("/")[-1], m) for m in COMPARE_MODELS] + [(MODEL_NAME, str(MODEL_DIR))]
rng_eval = np.random.default_rng(SEED + 1)
EVAL_ROWS = []
for country in EVAL_COUNTRIES:
    t_c = time.time()
    s_ids, s_texts, _, _, _ = read_texts(country_files(country, ("s1",)))
    c_ids, c_texts, c_native, c_empty, _ = read_texts(country_files(country, ("s2", "s3")), keep=QUICK_KEEP.get(country, frozenset()))
    s_pos = pd.Series(np.arange(len(s_ids)), index=s_ids)
    c_pos = pd.Series(np.arange(len(c_ids)), index=c_ids)
    p = PAIRS[PAIRS["s1"].isin(s_pos.index) & PAIRS["cand"].isin(c_pos.index)]
    with_match = set(p["s1"])
    unseen = np.array(sorted(s for s in LGBM_S1[country] if s in with_match), dtype=object)
    groups = {"unseen": unseen[rng_eval.permutation(len(unseen))[:EVAL_S1]]}
    used = USED_PAIRS.get(country, set())
    if used:
        seen = np.array(sorted({k.split("|")[0] for k in used}), dtype=object)
        groups["seen"] = seen[rng_eval.permutation(len(seen))[:SEEN_EVAL_S1]]
    q_ids = np.concatenate(list(groups.values()))
    q_group = np.concatenate([[g] * len(v) for g, v in groups.items()])
    tp = p[p["s1"].isin(set(q_ids))].reset_index(drop=True)
    q_index = pd.Series(np.arange(len(q_ids)), index=q_ids)
    tq = q_index.reindex(tp["s1"]).to_numpy().astype(np.int64)
    tc = c_pos.reindex(tp["cand"]).to_numpy().astype(np.int64)
    t_group = q_group[tq]
    if "seen" in groups:                                   # seen: only the pairs actually used in training
        was_used = (tp["s1"] + "|" + tp["cand"]).isin(used).to_numpy()
        keep = (t_group == "unseen") | was_used
        tq, tc, t_group = tq[keep], tc[keep], t_group[keep]
    slices = {"all": np.ones(len(tc), bool), "Indian-script name": c_native[tc], "empty address": c_empty[tc],
              "Latin, with address": ~c_native[tc] & ~c_empty[tc]}
    q_texts = [s_texts[i] for i in s_pos.reindex(q_ids).to_numpy()]
    print(f"{country}: {len(c_ids):,} S2/S3 records searched; {len(q_ids):,} S1 "
          f"({', '.join(f'{g} {len(v):,}' for g, v in groups.items())}); {len(tc):,} true pairs", flush=True)
    for label, name in EVAL_MODELS:
        try:
            enc = load_encoder(name)
        except Exception as e:                             # e.g. no internet for the e5 download
            print(f"  {label}: not evaluated ({type(e).__name__}: {e})")
            continue
        c_emb = embed_all(enc, c_texts, f"{country} {label} S2/S3")
        q_emb = embed_all(enc, q_texts, f"{country} {label} S1")
        del enc
        gc.collect()
        mu = c_emb.mean(axis=0, dtype=np.float64).astype(np.float32)[None, :]
        _, hits = exact_search(c_emb, q_emb, mu, max(EVAL_K))
        match = hits[tq] == tc[:, None]
        found_rank = np.where(match.any(axis=1), match.argmax(axis=1), max(EVAL_K))
        for g in groups:
            for s_name, s_mask in slices.items():
                m = (t_group == g) & s_mask
                if m.any():
                    EVAL_ROWS.append({"country": country, "S1": g, "true record": s_name, "model": label,
                                      "true pairs": int(m.sum()),
                                      **{f"recall@{k}": round(float((found_rank[m] < k).mean()), 4) for k in EVAL_K}})
        del c_emb, q_emb, hits, match
        gc.collect()
        if GPUS:
            torch.cuda.empty_cache()
    print(f"{country}: evaluated in {(time.time() - t_c) / 60:.1f} min")

EVAL = pd.DataFrame(EVAL_ROWS)
EVAL.to_csv(OUT_DIR / "eval_recall.csv", index=False)
if len(EVAL):
    labels = [lab for lab, _ in EVAL_MODELS if lab in set(EVAL["model"])]
    wide = EVAL.pivot_table(index=["country", "S1", "true record", "true pairs"], columns="model",
                            values=[f"recall@{k}" for k in EVAL_K])
    wide = wide.reindex(pd.MultiIndex.from_frame(EVAL[["country", "S1", "true record", "true pairs"]].drop_duplicates()))
    cols = []
    for k in EVAL_K:
        cols += [(f"recall@{k}", lab) for lab in labels]
        if len(labels) > 1 and MODEL_NAME in labels:
            wide[(f"recall@{k}", "gain")] = wide[(f"recall@{k}", MODEL_NAME)] - wide[(f"recall@{k}", labels[0])]
            cols.append((f"recall@{k}", "gain"))
    display(wide[cols].round(4))
else:
    print("no evaluation rows (no model could be loaded, or no held-out S1 with a match)")

## Reading the results, and what to do next
* **"unseen, all, recall@30" is the number that matters** (India's). It is the share of true pairs the model puts in
  the top 30 for S1 it never trained on. For reference, 02 full's train report for e5-small top-30 is 0.9471 (India).
* **Clearly above e5** (+0.005 or more): run 02 full with this model, then 03 full, as in the first cell. The e5
  score feeds the matcher's strongest features, so the end result can move more or less than recall.
* **Below e5:** try more of either stage (`MLM_TEXTS_PER_COUNTRY`, `MLM_EPOCHS`, `PAIRS_PER_COUNTRY`) or a bigger model
  (`LAYERS=12`). Rerun stage 2 alone with `MLM_MODEL=<attached>/er_embed/er-embed-small-mlm`.
* **"Indian-script name":** the slice where e5's general tokenizer is weakest. Our tokenizer was learned from these
  records. This is where an own model should gain first.
* **"seen" far above "unseen":** the model memorized the `matches` pairs. Use fewer, or a lower `CL_LR`.
* **France** has no labels: only the full pipeline and the leaderboard can say whether the noisy copies taught it
  enough.

In [ ]:
CONFIG = {"model_dir": str(MODEL_DIR), "mlm_dir": str(MLM_DIR), "quick_run": QUICK_RUN, "use_test_text": USE_TEST_TEXT,
          "vocab_size": VOCAB_SIZE, "hidden": HIDDEN, "layers": LAYERS, "heads": HEADS, "max_len": MAX_LEN,
          "tokenizer_texts": TOKENIZER_TEXTS, "mlm_texts_per_country": MLM_TEXTS_PER_COUNTRY, "mlm_epochs": MLM_EPOCHS,
          "mlm_batch": MLM_BATCH, "mlm_lr": MLM_LR, "pairs_per_country": PAIRS_PER_COUNTRY, "hard_neg_top": HARD_NEG_TOP,
          "noisy_pairs_per_country": NOISY_PAIRS_PER_COUNTRY, "script_pairs": SCRIPT_PAIRS, "cl_epochs": CL_EPOCHS,
          "cl_batch": CL_BATCH, "cl_lr": CL_LR, "scale": SCALE, "prefix": PREFIX, "lgbm_seed": LGBM_SEED,
          "lgbm_train_s1_per_country": LGBM_TRAIN_S1_PER_COUNTRY, "pairs": STATS, "stage1": MLM_SUMMARY,
          "stage2": CL_SUMMARY, "compare_models": COMPARE_MODELS, "word_map_words": len(WORDMAP),
          "processed_dir": str(PROCESSED_DIR), "bucket_dir": str(BUCKET_DIR),
          "sentence_transformers": sentence_transformers.__version__, "eval": EVAL_ROWS}
(OUT_DIR / "embed_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")
shutil.rmtree(WORK_DIR, ignore_errors=True)      # pair files, corpus, tokenized text: not needed any more
print(f"saved to {OUT_DIR}:", sorted(p.name for p in OUT_DIR.iterdir()))
print(f"next: 02 full with E5_MODELS=<attached dataset>/{OUT_DIR.name}/{MODEL_DIR.name}, then 03 full with "
      f"BUCKET_DIR=<that run>/embeddings_full/{MODEL_DIR.name}/buckets "
      f"(TRAIN_S1_PER_COUNTRY={LGBM_TRAIN_S1_PER_COUNTRY}, SEED={LGBM_SEED})")